In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from scipy import signal
from ipywidgets import Play, IntSlider, RadioButtons, HBox, VBox, Output, FloatSlider, jslink
from IPython.display import display, clear_output, HTML


# ============================================================
# 1. Signal Preparation
# ============================================================

fs, duration = 8000, 1.0
t = np.linspace(0, duration, int(fs * duration), endpoint=False)

carrier = signal.chirp(t, f0=150, f1=600, t1=duration, method='quadratic', phi=-90, vertex_zero=True)
noise = np.random.normal(0, 0.1, t.shape)
vocal_signal = carrier + noise


# ============================================================
# 2. STFT Parameters
# ============================================================

frame_len, hop_size = 300, 150


# ============================================================
# 3. Window Function
# ============================================================

def get_window(name):
    return {'Rectangle': np.ones(frame_len), 'Hamming': np.hamming(frame_len), 'Von Hann': np.hanning(frame_len), 'Bartlett': np.bartlett(frame_len)}[name]


# ============================================================
# 4. Initial STFT
# ============================================================

f, t_stft, Zxx = signal.stft(vocal_signal, fs=fs, window=get_window('Hamming'), nperseg=frame_len, noverlap=hop_size)
num_frames = len(t_stft)


# ============================================================
# 5. Output Area
# ============================================================

plot_output = Output()


# ============================================================
# 6. Figure
# ============================================================

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(17.25, 8), gridspec_kw={'height_ratios': [2.5, 2, 2]})
plt.subplots_adjust(hspace=0.4, right=0.90)


# ============================================================
# 7. Upper Plot — Fixed Window / Moving Signal
# ============================================================

ax1.set_title('Signal Passing Through a Fixed Analysis Window', fontsize=11)
ax1.set_ylabel('Amplitude', fontsize=10)
ax1.set_xlim(-0.25, 0.25)
ax1.set_ylim(-1.5, 1.5)
ax1.grid(True)

window_left, window_width = -frame_len / (2 * fs), frame_len / fs

fixed_window = Rectangle((window_left, -1.5), window_width, 3.0, color='red', alpha=0.25)
ax1.add_patch(fixed_window)

line_moving_signal, = ax1.plot([], [], color='black', alpha=0.65)

ax1.axvline(window_left, color='red', linestyle='--', alpha=0.7)
ax1.axvline(window_left + window_width, color='red', linestyle='--', alpha=0.7)


# ============================================================
# 8. Middle Plot — Current Frame
# ============================================================

line_frame, = ax2.plot([], [], color='black', label='Original')
line_windowed, = ax2.plot([], [], color='red', label='Windowed')

ax2.set_title('Signal Inside the Fixed Analysis Window', fontsize=11)
ax2.set_xlabel('Sample', fontsize=10)
ax2.set_ylabel('Amplitude', fontsize=10)
ax2.set_xlim(0, frame_len - 1)
ax2.set_ylim(-1.5, 1.5)
ax2.grid(True)

ax2.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=9)


# ============================================================
# 9. Lower Plot — Magnitude Spectrum
# ============================================================

line_spectrum, = ax3.plot([], [], color='red')

ax3.set_title('Magnitude Spectrum of Current Frame — Hamming Window', fontsize=11)
ax3.set_xlabel('Frequency (Hz)', fontsize=10)
ax3.set_ylabel('Magnitude', fontsize=10)
ax3.set_xlim(0, fs / 2)
ax3.set_ylim(0, 1.2)
ax3.grid(True)


# ============================================================
# 10. Window Selection
# ============================================================

window_selector = RadioButtons(options=['Rectangle', 'Hamming', 'Von Hann', 'Bartlett'], value='Hamming', description='Window:')
window_selector.layout = {'width': 'auto'}
window_selector.style = {'description_width': '60px'}
window_selector.add_class('horizontal-radio')


# ============================================================
# 11. Horizontal Radio Buttons
# ============================================================

display(HTML("""
<style>
.horizontal-radio .widget-radio-box {
    display: flex !important;
    flex-direction: row !important;
    gap: 12px !important;
    align-items: center !important;
}
.horizontal-radio .widget-radio-item { margin-right: 8px !important; }
.horizontal-radio { width: auto !important; }
</style>
"""))


# ============================================================
# 12. Animation Controls
# ============================================================

frame_slider = IntSlider(min=0, max=num_frames - 1, step=1, value=0, description='Frame:', continuous_update=True, style={'description_width': 'initial'}, layout={'width': '450px'})

play = Play(value=0, min=0, max=num_frames - 1, step=1, interval=150, description='Play', disabled=False)

speed = FloatSlider(value=150, min=50, max=500, step=10, description='Interval (ms):', continuous_update=True, layout={'width': '300px'})

jslink((play, 'value'), (frame_slider, 'value'))


# ============================================================
# 13. Update Function
# ============================================================

def update_frame(frame_index, window_name):

    current_window = get_window(window_name)

    f_current, t_current, Zxx_current = signal.stft(vocal_signal, fs=fs, window=current_window, nperseg=frame_len, noverlap=hop_size)

    center_sample = int(t_current[frame_index] * fs)
    start_sample = max(0, center_sample - frame_len // 2)
    end_sample = start_sample + frame_len

    if end_sample > len(t):
        end_sample, start_sample = len(t), max(0, len(t) - frame_len)

    visible_half = 0.5
    left_sample = max(0, start_sample - int(visible_half * fs))
    right_sample = min(len(t), end_sample + int(visible_half * fs))

    relative_t = -t[left_sample:right_sample] + t[start_sample] + frame_len / (2 * fs)

    line_moving_signal.set_data(relative_t, vocal_signal[left_sample:right_sample])

    current_segment = vocal_signal[start_sample:end_sample]

    if len(current_segment) < frame_len:
        current_segment = np.pad(current_segment, (0, frame_len - len(current_segment)), mode='constant')

    current_windowed = current_segment * current_window
    samples = np.arange(frame_len)

    line_frame.set_data(samples, current_segment)
    line_windowed.set_data(samples, current_windowed)

    current_spectrum = np.abs(Zxx_current[:, frame_index])
    line_spectrum.set_data(f_current, current_spectrum)

    ax2.set_title(f'Signal Inside the Fixed Analysis Window — {window_name}', fontsize=11)
    ax3.set_title(f'Magnitude Spectrum of Current Frame — {window_name} Window', fontsize=11)


# ============================================================
# 14. Frame Callback
# ============================================================

def frame_changed(change):
    if change['name'] == 'value':
        update_frame(change['new'], window_selector.value)
        with plot_output:
            clear_output(wait=True)
            display(fig)


frame_slider.observe(frame_changed, names='value')


# ============================================================
# 15. Window Callback
# ============================================================

def window_changed(change):
    if change['name'] == 'value':
        update_frame(frame_slider.value, change['new'])
        with plot_output:
            clear_output(wait=True)
            display(fig)


window_selector.observe(window_changed, names='value')


# ============================================================
# 16. Play / Pause Callback
# ============================================================

def play_changed(change):
    if change['name'] == 'playing':
        window_selector.disabled = change['new']


play.observe(play_changed, names='playing')


# ============================================================
# 17. Speed Callback
# ============================================================

def speed_changed(change):
    if change['name'] == 'value':
        play.interval = int(change['new'])


speed.observe(speed_changed, names='value')


# ============================================================
# 18. Initial Display
# ============================================================

update_frame(0, 'Hamming')

with plot_output:
    display(fig)

plt.close(fig)


# ============================================================
# 19. Final Layout
# ============================================================

window_row = HBox([window_selector])
animation_row = HBox([play, frame_slider, speed])

display(VBox([window_row, animation_row, plot_output]))